📘 Notebook — CISPC157 (Validación Archivo Asistencias Allianz Partners)

| **Campo** | **Descripción** |
|:---|:---|
| **Objetivo** | Validar, depurar y consolidar la información del archivo de Asistencias de Allianz Partners. El proceso identifica duplicados, valida campos obligatorios, cruza con pólizas vigentes, verifica tarifas/costos y controla los límites de servicios por póliza. Genera reportes de inconsistencias, registros válidos y siniestros instrumentales. |
| **Autor** | Juan Salazar Saenz |
| **Correo** | juan.salazar@externos.allianz.co |
| **Fecha de Creación** | 2026/02/03 |
| **Periodicidad** | Mensual (Antes del cierre) |

---

## 📝 Log de Versiones

| **Versión** | **Fecha** | **Autor** | **Descripción del Cambio** |
|:---:|:---:|:---|:---|
| 1.0 | 2026/02/03 | Juan Salazar Saenz | Migración inicial de SAS a PySpark (Azure Synapse). |

---

## 📖 Descripción General

Este notebook migra la lógica del proceso SAS **CISPC157**. El flujo de trabajo realiza las siguientes etapas:

1.  **Configuración de Fechas:** Determina el periodo de procesamiento (AAAAMM) basándose en la fecha de ejecución (si es antes o después del día 10).
2.  **Lectura y Depuración:** Carga el archivo de asistencias (`WORK.ASISTENCIAS`), elimina duplicados exactos y valida la integridad de campos obligatorios (Placa, Servicio, Ciudad, Valor, etc.).
3.  **Validación de Pólizas:** Cruza la información con la cartera de autos (`THUNICA_AUTOS_COL` y `CAR_INFOGENENERAL`) para determinar si el vehículo tenía una póliza vigente al momento del servicio.
4.  **Validación de Servicios y Tarifas:**
    *   Verifica que el servicio prestado sea válido según el catálogo (`AUX_SERVICIOSVALIDOS`).
    *   Compara los costos reportados contra las tarifas pactadas (`AUX_VALORSERVICIO`).
    *   Calcula el costo estimado por Allianz.
5.  **Control de Límites (Topes):** Analiza el histórico de servicios (`ASISTENCIAS_ALZ_PARTNERS`) junto con los nuevos registros para determinar si se excede el límite de eventos cubiertos por vigencia.
6.  **Generación de Salidas:**
    *   Separa los registros en **Válidos** e **Inconsistentes**.
    *   Actualiza la tabla histórica `ASISTENCIAS_ALZ_PARTNERS`.
    *   Genera archivos planos (CSV) para: Asistencias OK, Inconsistencias y Siniestros Instrumentales.
    *   Realiza la copia de reportes a las rutas de negocio correspondientes.

---

## 📥 Inventario de Datos

### **Inputs (Tablas Fuente)**
| **Nombre Tabla (SAS)** | **Nombre Tabla (Synapse)** | **Descripción** |
|:---|:---|:---|
| `WORK.ASISTENCIAS` | *DataFrame en Memoria* | Datos crudos del archivo de asistencias cargado previamente. |
| `RAMOAUT.THUNICA_AUTOS_COL` | `tdp_core_IM_SAS.THUNICA_AUTOS_COL` | Tabla maestra de pólizas de autos (vigencias, anulaciones). |
| `CARTERA.CAR_INFOGENENERAL` | `tdp_core_IM_SAS.CAR_INFOGENENERAL` | Información general de la cartera (fechas de vencimiento). |
| `GENINT.AUX_FASECOLDA` | `tdp_core_IM_SAS.AUX_FASECOLDA` | Catálogo de referencias Fasecolda. |
| `GENSIN.AUX_SERVICIOSVALIDOS_ASIST_AUTOS` | `tdp_core_IM_SAS.AUX_SERVICIOSVALIDOS_ASIST_AUTOS` | Maestro de servicios válidos por estatus y tipo. |
| `GENSIN.AUX_VALORSERVICIO_ASIST_AUTOS` | `tdp_core_IM_SAS.AUX_VALORSERVICIO_ASIST_AUTOS` | Tabla de tarifas y costos máximos permitidos. |
| `GENSIN.ASISTENCIAS_ALZ_PARTNERS` | `tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS` | Histórico acumulado de asistencias validadas. |
| `GENSIN.AUX_SUC_MEDIADOR_ASIST_AUTOS` | `tdp_core_IM_SAS.AUX_SUC_MEDIADOR_ASIST_AUTOS` | Auxiliar para mapeo de sucursales y mediadores. |

### **Outputs (Resultados)**
| **Nombre Archivo / Tabla** | **Formato** | **Descripción** |
|:---|:---|:---|
| `tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS` | Tabla Delta/Parquet | Actualización del histórico con los nuevos registros validados. |
| `Asistencias_OK_Validaciones_AllianzPartners_YYYYMM.csv` | CSV (Delimitado ';') | Reporte de asistencias que pasaron todas las validaciones. |
| `Inconsistencias_Asistencias_AllianzPartners_YYYYMM.csv` | CSV (Delimitado ';') | Reporte de registros rechazados (duplicados, sin póliza, costo excedido, etc.). |
| `Siniestros_Instrumentales_Asistencia_AllianzPartners_YYYYMM.csv` | CSV (Delimitado ';') | Archivo para la generación de siniestros instrumentales contables. |

---

## 🔗 Dependencias (Macros)
Este proceso utiliza las siguientes macros de utilidad (ya migradas o integradas):
*   `co_copia_reportes`: Para mover los archivos generados a las carpetas de salida finales.


##📘 Código Global — CISPC157 (Validación Asistencias Allianz Partners)
Este bloque contiene la migración completa del proceso SAS `CISPC157`. 
El flujo incluye:
1.  **Configuración de Fechas:** Cálculo de `PROCESO_AAAAMM` según reglas de negocio (día >= 10).
2.  **Lectura de Datos:** Carga del archivo de asistencias (`df_asistencias`) y tablas maestras del Pool (`THUNICA`, `CARTERA`, `AUX_...`).
3.  **Validaciones:**
    *   Duplicidad.
    *   Campos obligatorios.
    *   Vigencia de Póliza (Cruce con Cartera).
    *   Validación de Servicios y Tarifas (Cruce con Maestros).
4.  **Lógica de Negocio Compleja:** Cálculo de vigencias dinámicas y control de límites de frecuencia (conteo de servicios históricos + actuales).
5.  **Generación de Salidas:**
    *   Persistencia de registros válidos en el Pool.
    *   Generación de reportes CSV (Inconsistencias, OK, Siniestros Instrumentales).

## 📑 Reporte de Lógica y Reglas de Negocio - Proceso CISPC157

Este documento detalla las validaciones, tablas y reglas aplicadas en el proceso de **Validación del Archivo de Asistencias Allianz Partners** (CISPC157).

---

## 1. Inventario de Tablas Involucradas

El proceso utiliza diversas fuentes de datos para ejecutar las validaciones técnicas, de negocio y de tarifas.

| Categoría | Tabla SAS | Origen / Tabla Synapse | Propósito |
| :--- | :--- | :--- | :--- |
| **Entrada Principal** | `WORK.ASISTENCIAS` | Archivo Plano (Allianz Partners) | Datos crudos del proveedor con los servicios de asistencia prestados. |
| **Cartera Autos** | `RAMOAUT.THUNICA_AUTOS_COL` | `tdp_core_IM_SAS.THUNICA_AUTOS_COL` | Maestro unificado de pólizas de autos para validación de vigencias y certificados. |
| **Cartera General** | `CARTERA.CAR_INFOGENENERAL` | `tdp_core_IM_SAS.CAR_INFOGENENERAL` | Información general de pólizas para cruce de estados y fechas de anulación. |
| **Catálogos** | `GENINT.AUX_FASECOLDA` | `tdp_core_IM_SAS.AUX_FASECOLDA` | Referencia de vehículos por código Fasecolda para clasificación técnica. |
| **Catálogos** | `GENSIN.AUX_VALORSERVICIO_ASIST_AUTOS` | `tdp_core_IM_SAS.AUX_VALORSERVICIO_ASIST_AUTOS` | Matriz de tarifas pactadas para validar que el costo del proveedor sea correcto. |
| **Catálogos** | `GENSIN.AUX_SERVICIOSVALIDOS_ASIST_AUTOS` | `tdp_core_IM_SAS.AUX_SERVICIOSVALIDOS_ASIST_AUTOS` | Catálogo de servicios permitidos por tipo de vehículo y cobertura. |
| **Control** | `GENSIN.AUX_CONTEOS_ASISTENCIAS_OFERTA` | `tdp_core_IM_SAS.AUX_CONTEOS_ASISTENCIAS_OFERTA` | Tabla de control para limitar el número de servicios permitidos por póliza/oferta. |

---

## 2. Reglas de Validación (Business Rules)

A continuación se describen las reglas aplicadas a cada registro del archivo de entrada:

### 🛠️ Validaciones Técnicas

#### **R1. Registros Duplicados**
* **Descripción:** Identifica si un servicio ha sido reportado más de una vez para evitar pagos dobles.
* **Campos de validación:** `expediente`, `placa`, `servicio`, `subservicio`, `fecha_apertura`, `hora_apertura` y `valor_total`. `local carretero`
* **Acción:** Se extraen a la tabla de inconsistencias con el motivo `REPETIDOS`.

#### **R2. Integridad de Datos (Campos Obligatorios)**
* **Descripción:** Garantiza que la información mínima necesaria para el procesamiento esté presente.
* **Validación:** El registro se rechaza si los campos `placa`, `servicio`, `ciudad_origen`, `ciudad_destino` o `valor_total` están vacíos.
* **Acción:** Se extraen con el motivo `CAMPOS_OBLIGATORIOS`.

---

### 💼 Validaciones de Negocio

#### **R3. Validación de Póliza Vigente**
* **Descripción:** Cruza el servicio con la cartera de Allianz para verificar que el vehículo tuviera cobertura en la fecha del evento.
* **Lógica:** La `Fecha_Apertura` del servicio debe estar en el rango: `FEFECTO <= Fecha <= MIN(FANUL, FIANUL, FVENCIM)`.
* **Acción:** Si no se encuentra póliza activa, se marca como `SIN POLIZA VIGENTE`.


### 🔍 Regla 3: Validación de Póliza Vigente (Cruce con Cartera)

Esta regla es el corazón del proceso. Su objetivo es verificar que el vehículo atendido por el proveedor tuviera una póliza activa en Allianz al momento de la prestación del servicio.

#### 📋 Tablas Involucradas

| Tabla | Ubicación / Fuente | Rol en la Validación |
| :--- | :--- | :--- |
| **`ServiciosCompletos`** | DataFrame Temporal | Registros del proveedor que ya superaron las validaciones técnicas iniciales. |
| **`THUNICA_AUTOS_COL`** | `tdp_core_IM_SAS` | Maestro de autos. Aporta fechas de efecto, terminación y estados de anulación. |
| **`CAR_INFOGENENERAL`** | `tdp_core_IM_SAS` | Cartera general. Aporta la fecha de vencimiento (`FECHVTO`) para validar periodos. |

---

#### 💻 Lógica de Negocio (Query de Validación)

El cruce se realiza mediante un `LEFT JOIN` entre los servicios y la cartera consolidada, aplicando filtros de fecha para asegurar la vigencia técnica:

```sql
-- Lógica equivalente en SQL para validar vigencias
SELECT 
    asist.*,
    cart.poliza,
    cart.fefecto,
    cart.fanul,
    cart.fianul,
    cart.motanul,
    cart.fechvto
FROM ServiciosCompletos AS asist
LEFT JOIN (
    -- Consolidación de Cartera Vigente
    SELECT 
        A.placa, A.poliza, A.aplica, A.fefecto, A.fterm, 
        A.fanul, A.fianul, A.motanul, B.fechvto
    FROM tdp_core_IM_SAS.THUNICA_AUTOS_COL A
    LEFT JOIN tdp_core_IM_SAS.CAR_INFOGENENERAL B
        ON A.POLIZA = B.POLIZA AND A.APLICA = B.APLICA
    WHERE A.O_CAR = 1 AND A.RAMO <> 1311
) AS cart
ON asist.PLACA = cart.PLACA
   -- Criterios de Vigencia Temporal:
   AND asist.Fecha_Apertura >= cart.FEFECTO 
   AND (
       cart.FANUL IS NULL                          -- Póliza activa sin anulación
       OR asist.Fecha_Apertura <= cart.FIANUL      -- Servicio antes de la anulación técnica
       OR cart.MOTANUL = 'Z'                       -- Motivo Z: Mantiene cobertura
       OR (cart.MOTANUL = 'Y' AND asist.Fecha_Apertura <= cart.FANUL) -- Motivo Y: Cobertura hasta fecha de anulación
   )
```


#### **R4. Clasificación Técnica del Vehículo**
* **Descripción:** Determina si el vehículo es **Liviano** o **Pesado** basándose en el código del servicio.
* **Lógica:** * Si el código inicia con `L`, `M` o `T` → **Liviano**.
    * Si el código inicia con `A`, `B` o `P` → **Pesado**.


### 🚗 Regla 4: Clasificación Técnica (Livianos vs Pesados)

Esta regla categoriza el tipo de vehículo atendido basándose en la codificación interna del servicio. Esta clasificación es el "puente" para poder validar si el costo cobrado por el proveedor corresponde a la tarifa pactada para ese tipo de vehículo.

#### 📋 Lógica de Clasificación

El proceso evalúa el primer carácter del campo `servicio` (o `cod_servicio`) reportado por el proveedor:

| Categoría | Prefijos de Servicio | Tipos de Vehículo Incluidos |
| :--- | :--- | :--- |
| **Liviano** | `L`, `M`, `T` | Automóviles, Motos, Camperos, Pick-ups. |
| **Pesado** | `A`, `B`, `P` | Camiones, Buses, Vehículos de carga pesada. |

---

#### 💻 Implementación en PySpark

En la migración, esta lógica se implementa mediante una columna condicional (`when().otherwise()`):

```python

# Creación de la columna de clasificación técnica
df_procesado = df_asistencias.withColumn(
    "liviano_pesado",
    F.when(F.substring(F.col("servicio"), 1, 1).isin("L", "M", "T"), "Liviano")
     .when(F.substring(F.col("servicio"), 1, 1).isin("A", "B", "P"), "Pesado")
     .otherwise("No Clasificado")
)
```

#### **R5. Validación de Tarifas y Servicios**
* **Descripción:** Verifica que el tipo de servicio sea válido y que el cobro no supere el tope máximo configurado.
* **Cruce:** Se valida contra `AUX_VALORSERVICIO_ASIST_AUTOS` usando `servicio`, `subservicio` y la clasificación (Liviano/Pesado).
* **Acción:** Si el valor excede lo pactado o el servicio no existe para ese tipo de vehículo, se genera una alerta.


### 💰 Regla 5: Validación de Servicios y Tarifas

Esta regla asegura que el servicio reportado por el proveedor esté dentro del catálogo autorizado por Allianz y que los costos aplicados sean los correctos según la matriz de negociación.

#### 📋 Tablas Involucradas

| Tabla | Fuente / Origen | Rol en la Validación |
| :--- | :--- | :--- |
| **`POL_ASISTENCIA_FASECOLDA`** | Interna (Paso anterior) | Registros con póliza vigente y clasificación Liviano/Pesado ya asignada. |
| **`AUX_SERVICIOSVALIDOS_`** | `tdp_core_IM_SAS` | Catálogo maestro de combinaciones permitidas de servicio, subservicio y tipo de vehículo. |
| **`AUX_VALORSERVICIO_`** | `tdp_core_IM_SAS` / Excel | Matriz de tarifas pactadas (Banderazos, costos por km, etc.). |

---

#### 🛠️ Lógica de Validación

La validación se divide en dos componentes críticos:

**1. Existencia del Servicio (Catálogo):**
Se verifica que la combinación de campos exista en el catálogo maestro. Si no existe, el registro se rechaza.
* **Llave de Cruce:** `servicio`, `subservicio`, `local_carretero`, `estatus` y `liviano_pesado`.
* **Error:** `SERVICIO_NO_VALIDO`.

**2. Cálculo de Periodo de Vigencia:**
El proceso calcula el "Intervalo de Vigencia" (Año 1, Año 2, etc.) comparando la `Fecha_Apertura` del servicio con el inicio de la póliza (`FEFECTO`) y el vencimiento (`FECHVTO`). Esto es crucial para la **Regla 6** (Control de topes por año).

```python
# Lógica conceptual del cálculo de vigencia (Intervalo)
if fecha_vencimiento == fecha_termino:
    inicio_vigencia = fecha_efecto
    fin_vigencia = fecha_termino
else:
    # Lógica para pólizas con renovaciones o extensiones
    # Se calcula el intervalo anual donde cae la Fecha_Apertura
```


#### **R6. Control de Ofertas (Contador de Servicios)**
* **Descripción:** Algunas ofertas comerciales limitan el número de servicios (ej. máximo 2 grúas al año). 
* **Lógica:** Consulta `AUX_CONTEOS_ASISTENCIAS_OFERTA` para verificar si la póliza ya agotó sus cupos.

### 📊 Regla 6: Control de Topes (Contador de Servicios por Oferta)

Esta regla protege la siniestralidad de la compañía verificando que el proveedor no esté cobrando servicios por encima del límite estipulado en la oferta comercial de la póliza.

#### 📋 Tablas Involucradas

| Tabla | Fuente / Origen | Rol en la Validación |
| :--- | :--- | :--- |
| **`Intervalo_Vigencia`** | Interna (Regla 5) | Contiene los registros actuales situados en su año de vigencia correspondiente. |
| **`ASISTENCIAS_ALZ_PARTNERS`** | `tdp_core_IM_SAS` | Repositorio histórico. Se usa para contar cuántos servicios ha tenido la póliza en el pasado dentro de la misma vigencia. |
| **`param_serv_autos`** | Archivo / Tabla Parámetro | Define el `Limite_de_eventos` permitido para cada combinación de servicio/subservicio/tipo de vehículo. |

---

#### 🛠️ Lógica de Validación (Contador)

El proceso realiza un conteo acumulativo:
1.  **Conteo Histórico:** Suma los servicios ya pagados en meses anteriores que pertenecen al mismo intervalo de vigencia de la póliza.
2.  **Conteo Actual:** Suma los servicios que vienen en el archivo nuevo del proveedor.
3.  **Comparación:**
    * `Total_Servicios = (Historico + Actual)`
    * Si `Total_Servicios > Limite_de_eventos`, el registro se marca como excedido.

#### 💻 Query de Agregación (PySpark)

```python
# Agregación para contar servicios por Póliza e Intervalo de Vigencia
df_conteo = df_unificado.groupBy(
    "poliza", "servicio", "vigencia", "liviano_pesado"
).agg(
    F.count("expediente").alias("servicios_consumidos")
)

# Cruce con la tabla de límites (parámetros)
df_validado = df_conteo.join(df_parametros_topes, on=["servicio", "liviano_pesado"], how="left") \
    .withColumn("Excede_Tope", F.col("servicios_consumidos") > F.col("Limite_de_eventos"))




In [145]:
%run /oficina_datos_ext/generico/nbk_generico


In [146]:
"""
## Descripción
Esta celda realiza la configuración inicial de fechas de proceso, lectura y saneamiento de los insumos de asistencias (Allianz) y la preparación de la base de cartera consolidando información de THUNICA e INFOGENERAL.

Entrada
- **fecha_ejecucion (str)**: Fecha de ejecución del proceso en formato `YYYYMMDD`.
- **container / storage_curated (variables)**: Parámetros del entorno para la ruta de Azure Data Lake.
- **tdp_core_IM_SAS.THUNICA_AUTOS_COL (tabla)**: Tabla de origen con información de pólizas de autos.
- **tdp_core_IM_SAS.CAR_INFOGENENERAL (tabla)**: Tabla de origen con fechas de vencimiento de cartera.

Salida
- **df_servicios_completos (DataFrame)**: Registros del reporte de asistencias que pasaron las validaciones técnicas (sin duplicados y con campos obligatorios).
- **df_servicios_repetidos (DataFrame)**: Registros rechazados por duplicidad.
- **df_servicios_incompletos (DataFrame)**: Registros rechazados por falta de datos obligatorios.
- **df_cartera (DataFrame)**: Vista consolidada de cartera para cruces posteriores.
- **PROCESO_AAAAMM (str)**: Periodo de proceso calculado en formato `YYYYMM`.
"""


try:
    logger.info("Iniciando Chunk 1: Configuración de fechas y validaciones técnicas")

    # 1. Lógica de Fechas
    fecha_ejecucion = "20260131" 
    fechas_parametricas = calcula_fecha(fecha_ejecucion)

    fecha_sql = f"{fecha_ejecucion[0:4]}-{fecha_ejecucion[4:6]}-{fecha_ejecucion[6:8]}"
    
    year_param = int(fecha_ejecucion[0:4])
    month_param = int(fecha_ejecucion[4:6])
    day_param = int(fecha_ejecucion[6:8]) if len(fecha_ejecucion) >= 8 else 1
    
    fecha_base = datetime.date(year_param, month_param, day_param)
    dia_actual_sistema = datetime.datetime.now().day
    
    if dia_actual_sistema >= 10:
        fecha_calculada = fecha_base + relativedelta(months=1)
    else:
        fecha_calculada = fecha_base

    logger.info(f"Fecha calculado {fecha_calculada}")

    PROCESO_AAAAMM = fecha_calculada.strftime("%Y%m")
    first_month_date_str = fecha_calculada.replace(day=1).strftime("%Y-%m-%d")
    
    logger.info(f"Periodo de proceso determinado: {PROCESO_AAAAMM}")
 
 
      # 2. Lectura del Archivo
    logger.info("Leyendo reporte juan...")
    ruta_insumo = f"abfss://{container}@{storage_curated}.dfs.core.windows.net/Entradas/vp_financiera/gerencia_contabilidad_tributaria/"
    nombre_archivo_asistencias = f"ALLIANZ_AZP_{PROCESO_AAAAMM}.csv" 

    
    logger.info(f"Configurando lectura de asistencias desde: {ruta_insumo}/{nombre_archivo_asistencias}")

    df_asistencias = leer_archivo(
        ruta=ruta_insumo,
        archivo=nombre_archivo_asistencias,
        delimitador=";", 
        cabecera=True,
        hoja=None,
        encoding="ISO-8859-1"
    )
    
    if df_asistencias is None:
        raise ValueError("El DataFrame de asistencias no se pudo cargar (es None).")

    logger.info("================ VALIDACIÓN ASISTENCIAS / LECTURA ================")
    logger.info("SAS Log (WORK.ASISTENCIAS): Esperado 9491 registros.")
    logger.info(f"PySpark (df_asistencias): {df_asistencias.count()} registros.")
    logger.info("======================================================")

    # 3. Preparación y Saneado de Valor_Total
    # Se agrega limpieza de caracteres para que el cast a double no devuelva null
    df_asist_prep = df_asistencias.withColumn(
        "Fecha_Apertura_Dt", to_date(col("Fecha_Apertura"))
    ).withColumn(
        "Valor_Total", 
        regexp_replace(regexp_replace(col("Valor_Total"), r"[\$\s\.]", ""), r",", ".").cast("double")
    )

    logger.info("================ VALIDACIÓN ASIST_PREP / FACTURACIONMESSORT ================")
    logger.info("SAS Log (WORK.FACTURACIONMESSORT): Esperado 9491 registros.")
    logger.info(f"PySpark (df_asist_prep): {df_asist_prep.count()} registros.")
    logger.info("======================================================")

    # R1: Identificar Registros Repetidos
    cols_clave_dup = ["expediente", "placa", "servicio", "subservicio", "fecha_apertura", "hora_apertura", "Valor_Total"]
    window_dup = Window.partitionBy(*cols_clave_dup)
    
    df_con_conteo = df_asist_prep.withColumn("cnt_duplicados", count("*").over(window_dup))

    df_servicios_repetidos = df_con_conteo.filter(col("cnt_duplicados") > 1) \
        .withColumn("Inconsistencia", lit("REPETIDOS")) \
        .withColumn("Desc_Inconsistencia", concat(
            lit("expediente:"), col("expediente"), 
            lit(" | placa:"), col("placa"), 
            lit(" | servicio:"), col("servicio"), 
            lit(" | Valor_Total:"), col("Valor_Total")
        ))

    df_sin_repetidos = df_con_conteo.filter(col("cnt_duplicados") == 1).drop("cnt_duplicados")
    
    logger.info("================ VALIDACIÓN SERVICIOSSINREPETIDOS / R1 ================")
    logger.info("SAS Log (WORK.SERVICIOSSINREPETIDOS): Esperado 9491 registros.")
    logger.info(f"PySpark (df_sin_repetidos): {df_sin_repetidos.count()} registros.")
    logger.info("======================================================")

    # R2: Validación de Campos Obligatorios
    cond_incompletos = (
        col("placa").isNull() | (col("placa") == "") |
        col("servicio").isNull() | (col("servicio") == "") |
        col("Ciudad_de_Origen").isNull() | (col("Ciudad_de_Origen") == "") |
        col("Ciudad_de_Destino").isNull() | (col("Ciudad_de_Destino") == "") |
        col("Valor_Total").isNull() |
        col("Local_carretero").isNull() | (col("Local_carretero") == "")
    )

    df_servicios_incompletos = df_sin_repetidos.filter(cond_incompletos) \
        .withColumn("Inconsistencia", lit("CAMPOS_OBLIGATORIOS")) \
        .withColumn("Desc_Inconsistencia", concat(
            when(col("placa").isNull() | (col("placa") == ""), lit(" -placa-")).otherwise(lit("")),
            when(col("servicio").isNull() | (col("servicio") == ""), lit(" -servicio-")).otherwise(lit("")),
            when(col("Valor_Total").isNull(), lit(" -Valor_Total-")).otherwise(lit(""))
        ))

    logger.info("================ VALIDACIÓN SERVICIOSINCOMPLETOS / R2 ================")
    logger.info("SAS Log (WORK.SERVICIOSINCOMPLETOS): Esperado 0 registros.")
    logger.info(f"PySpark (df_servicios_incompletos): {df_servicios_incompletos.count()} registros.")
    logger.info("======================================================")

    df_servicios_completos = df_sin_repetidos.filter(~cond_incompletos)
    
    logger.info("================ VALIDACIÓN SERVICIOSCOMPLETOS ================")
    logger.info("SAS Log (WORK.SERVICIOSCOMPLETOS): Esperado 9491 registros.")
    logger.info(f"PySpark (df_servicios_completos): {df_servicios_completos.count()} registros.")
    logger.info("======================================================")
    

    # 4. Preparación de Cartera
    query_thunica = f"""
        SELECT placa, poliza, aplica, ramo, servicio, modelo, sucursal, 
               codfasecolda, fefecto, fterm, fanul, fianul, motanul
        FROM tdp_core_IM_SAS.THUNICA_AUTOS_COL
        WHERE fecha = '{fecha_sql}'
          AND o_car = 1 AND ramo <> 1311
    """
    df_previgentes = consulta_extraccion_pool(query_thunica).dropDuplicates()

    logger.info("================ VALIDACIÓN PREVIGENTES ================")
    logger.info("SAS Log (WORK.PREVIGENTES): Esperado 1039402 registros.")
    logger.info(f"PySpark (df_previgentes): {df_previgentes.count()} registros.")
    logger.info("======================================================")

    query_infogen = f"""SELECT poliza, aplica, fechvto 
                        FROM tdp_core_IM_SAS.CAR_INFOGENENERAL
                        WHERE fecha = '{fecha_sql}'
                        and AGRUPACM in (1010,1040)"""
    df_infogen = consulta_extraccion_pool(query_infogen).dropDuplicates()

    logger.info("================ VALIDACIÓN CAR_INFOGENENERAL ================")
    logger.info("SAS Log (WORK.CAR_INFOGENERAL filtrado): Esperado 1099695 registros.")
    logger.info(f"PySpark (df_infogen): {df_infogen.count()} registros.")
    logger.info("======================================================")

    df_cartera = df_previgentes.alias("thu").join(
        df_infogen.alias("info"),
        (col("thu.poliza") == col("info.poliza")) & (col("thu.aplica") == col("info.aplica")),
        "left"
    ).select(
        col("thu.placa"), col("thu.poliza"), col("thu.aplica"),
        col("thu.ramo").alias("ramo_Num"),
        col("thu.servicio").alias("cod_servicio"),
        col("thu.modelo"), col("thu.sucursal"),
        col("thu.codfasecolda").alias("cod_fasecolda"),
        to_date(col("thu.fefecto")).alias("fefecto"),
        to_date(col("thu.fterm")).alias("fterm"),
        to_date(col("thu.fanul")).alias("fanul"),
        to_date(col("thu.fianul")).alias("fianul"),
        col("thu.motanul"),
        to_date(col("info.fechvto")).alias("fechvto")
    )

    logger.info("================ VALIDACIÓN CARTERA ================")
    logger.info("SAS Log (WORK.CARTERA): Esperado 1039402 registros.")
    logger.info(f"PySpark (df_cartera): {df_cartera.count()} registros.")
    logger.info("======================================================")
    
    
    logger.info("Chunk 1 finalizado exitosamente.")

except Exception as e:
    logger.error(f"Error en Chunk 1: {str(e)}")
    raise e


In [147]:
"""
## Descripción
Esta celda ejecuta el cruce de los servicios de asistencias contra la base de cartera para asignar pólizas vigentes. Además, valida la información técnica del vehículo mediante el catálogo de Fasecolda, verifica la validez de los servicios prestados y calcula los intervalos de vigencia anual para cada registro.

Entrada
- **df_servicios_completos (DataFrame)**: Insumo de asistencias saneado proveniente del Chunk 1.
- **df_cartera (DataFrame)**: Base de cartera preparada en el Chunk 1.
- **tdp_core_IM_SAS.AUX_FASECOLDA (tabla)**: Catálogo maestro de vehículos (Marca, Clase, Tipo).
- **tdp_core_IM_SAS.AUX_SERVICIOSVALIDOS_ASIST_AUTOS (tabla)**: Matriz de validación de servicios permitidos.

Salida
- **df_servicios_validos (DataFrame)**: Registros con póliza vigente, información de Fasecolda y servicios validados.
- **df_sin_poliza_cruce / df_norenovadas (DataFrames)**: Registros rechazados por falta de coincidencia o vigencia en cartera.
- **df_servicios_invalidos (DataFrame)**: Registros cuya combinación de servicio/estatus no existe en el maestro.
- **Campos calculados**: `liviano_pesado`, `Fecha_Inicio`, `Fecha_Fin` y `Vigencia`.
"""

try:
    logger.info("Iniciando Chunk 2: Cruce con Cartera y Validación de Servicios (Saneado de Raíz)")

    # 1. Limpieza inicial y preparación
    # Tip: Asegúrate de que df_servicios_completos tenga datos antes de seguir
    conteo_inicial_ch2 = df_servicios_completos.count()
    logger.info(f"Registros recibidos del Chunk 1: {conteo_inicial_ch2}")

    df_servicios_limpios = df_servicios_completos.withColumn(
        "Fecha_Apertura_Dt", to_date(col("Fecha_Apertura"), "d/M/yyyy")
    ).withColumn(
        "Valor_Total_Num", regexp_replace(regexp_replace(col("Valor_Total"), r"[\$\s]", ""), r"\.", "").cast("double")
    )

    # 2. Cruce con Cartera (Saneado de PLACA)
    logger.info("Realizando cruce con Cartera (df_cartera)...")
    ser_a = df_servicios_limpios.alias("ser")
    cart_a = df_cartera.alias("cart")
    
    condicion_vigencia = (
        (col("ser.placa") == col("cart.placa")) & 
        (col("ser.Fecha_Apertura_Dt") >= col("cart.fefecto")) &
        (col("cart.fanul").isNull() | (col("ser.Fecha_Apertura_Dt") <= col("cart.fianul")) | (col("cart.motanul") == "Z") |
        ((col("cart.motanul") == "Y") & (col("ser.Fecha_Apertura_Dt") <= col("cart.fanul"))))
    )

    df_priorizado = ser_a.join(cart_a, condicion_vigencia, "left_outer") \
        .select("ser.*", col("cart.poliza"), col("cart.aplica"), col("cart.ramo_Num"), col("cart.cod_servicio"),
                col("cart.modelo"), col("cart.sucursal"), col("cart.cod_fasecolda"), col("cart.fefecto"),
                col("cart.fterm"), col("cart.fanul"), col("cart.fianul"), col("cart.motanul"), col("cart.fechvto")) \
        .withColumn("rn_prioridad", row_number().over(Window.partitionBy("expediente").orderBy(col("fterm").desc_nulls_last(), col("fanul").desc_nulls_last()))) \
        .filter(col("rn_prioridad") == 1).drop("rn_prioridad")

    logger.info(f"Registros df_priorizado: {df_priorizado.count()}")
   
    # 3. Clasificación de Vigencia
    df_con_poliza = df_priorizado.filter(col("poliza").isNotNull())
    df_sin_poliza_cruce = df_priorizado.filter(col("poliza").isNull())
    
    logger.info(f"Registros que cruzaron con cartera: {df_con_poliza.count()}")
    logger.info(f"Registros sin cruce inicial (Sin Póliza): {df_sin_poliza_cruce.count()}")

    cond_vigencia_final = (
        (col("fanul").isNull() & (col("Fecha_Apertura_Dt") <= col("fechvto"))) | 
        col("fanul").isNull() |
        (col("Fecha_Apertura_Dt") <= col("fianul")) | 
        (col("motanul") == "Z") |
        ((col("motanul") == "Y") & (col("Fecha_Apertura_Dt") <= col("fanul")))
    )

    df_polizas_vigentes = df_con_poliza.filter(cond_vigencia_final).withColumn("liviano_pesado", 
        when(substring(col("cod_servicio"), 1, 1).isin("L", "M", "T"), lit("Liviano"))
        .when(substring(col("cod_servicio"), 1, 1).isin("A", "B", "P"), lit("Pesado"))
        .otherwise(lit("No Clasificado")))

    df_norenovadas = df_con_poliza.filter(~cond_vigencia_final).withColumn("Inconsistencia", lit("SIN POLIZA VIGENTE")) \
        .withColumn("Desc_Inconsistencia", concat(lit("POLIZA_APLICA:"), col("poliza"), lit("-"), col("aplica")))

    logger.info(f"Pólizas vigentes final: {df_polizas_vigentes.count()}")
    logger.info(f"Pólizas no vigentes/renovadas: {df_norenovadas.count()}")

    # 4. Cruce con Catálogos (Validación de Servicios)
    logger.info("Cargando catálogos de Fasecolda y Servicios Válidos...")
    df_fasecolda = consulta_extraccion_pool("SELECT COD_FASECOLDA, MARCA, CLASE, TIPO, VERSION FROM tdp_core_IM_SAS.AUX_FASECOLDA WHERE 1=1").dropDuplicates()

    logger.info(f"Registros tdp_core_IM_SAS.AUX_FASECOLDA: {df_fasecolda.count()}")

    df_maestro_serv = consulta_extraccion_pool("SELECT SERVICIO, SUBSERVICIO, LOCAL_CARRETERO, ESTATUS, LIVIANO_PESADO FROM tdp_core_IM_SAS.AUX_SERVICIOSVALIDOS_ASIST_AUTOS WHERE 1=1").dropDuplicates()

    logger.info(f"Registros tdp_core_IM_SAS.AUX_SERVICIOSVALIDOS_ASIST_AUTOS: {df_maestro_serv.count()}")
    
    df_pol_fasecolda = df_polizas_vigentes.alias("p").join(
        df_fasecolda.alias("f"), col("p.cod_fasecolda") == col("f.COD_FASECOLDA"), "left"
    ).select("p.*", "f.MARCA", "f.CLASE", "f.TIPO", "f.VERSION")

    logger.info("Validando servicios contra Maestro de Servicios...")
    df_validar_cat = df_pol_fasecolda.alias("pol").join(
        df_maestro_serv.alias("m"),
        (col("pol.servicio") == col("m.SERVICIO")) & 
        (col("pol.subservicio") == col("m.SUBSERVICIO")) &
        (col("pol.local_carretero") == col("m.LOCAL_CARRETERO")) & 
        (col("pol.estatus") == col("m.ESTATUS")) &
        (col("pol.liviano_pesado") == col("m.LIVIANO_PESADO")),
        "left_outer"
    ).select(col("pol.*"), col("m.SERVICIO").alias("check_catalogo"))

    logger.info(f"Servicios válidos primer cruce: {df_pol_fasecolda.count()}")
    logger.info(f"Servicios válidos segundo cruce: {df_validar_cat.count()}")

    df_servicios_invalidos = df_validar_cat.filter(col("check_catalogo").isNull()) \
        .withColumn("Inconsistencia", lit("SERVICIO_NO_VALIDO")) \
        .withColumn("Desc_Inconsistencia", concat(lit("servicio:"), col("servicio"), lit(" | estatus:"), col("estatus")))

    df_pre_servicios_validos = df_validar_cat.filter(col("check_catalogo").isNotNull()).drop("check_catalogo")

    logger.info(f"Servicios válidos: {df_pre_servicios_validos.count()}")
    logger.info(f"Servicios inválidos (Rechazados): {df_servicios_invalidos.count()}")

    # 5. Cálculo de Vigencia e Intervalos
    logger.info("Calculando intervalos de vigencia (Fecha_Inicio / Fecha_Fin)...")
    df_servicios_validos = df_pre_servicios_validos.withColumn("Fecha_Inicio", expr("""
            CASE WHEN fechvto = fterm THEN fefecto
                 WHEN fechvto > fterm THEN
                    CASE WHEN month(fechvto) <= month(Fecha_Apertura_Dt) THEN
                            CASE WHEN (year(fechvto) - year(Fecha_Apertura_Dt)) <= 1 THEN add_months(fechvto, -12)
                                 ELSE add_months(fechvto, -12 * (year(fechvto) - year(Fecha_Apertura_Dt))) END
                         ELSE add_months(fechvto, -12 * ((year(fechvto) - year(Fecha_Apertura_Dt)) + 1)) END
            ELSE NULL END""")) \
        .withColumn("Fecha_Fin", expr("""
            CASE WHEN fechvto = fterm THEN fterm
                 WHEN fechvto > fterm THEN
                    CASE WHEN month(fechvto) <= month(Fecha_Apertura_Dt) THEN
                            CASE WHEN (year(fechvto) - year(Fecha_Apertura_Dt)) <= 1 THEN fechvto
                                 ELSE add_months(fechvto, -12 * (year(fechvto) - year(Fecha_Apertura_Dt)) + 12) END
                         ELSE add_months(fechvto, -12 * (year(fechvto) - year(Fecha_Apertura_Dt))) END
            ELSE NULL END""")) \
        .withColumn("Vigencia", concat(lit("Intervalo Vigencia "), year(col("Fecha_Inicio")), lit("-"), year(col("Fecha_Fin"))))

    logger.info(f"Chunk 2 finalizado exitosamente. Registros finales listos: {df_servicios_validos.count()}")

except Exception as e:
    logger.error(f"Error en Chunk 2: {str(e)}")
    raise e

In [148]:
"""
## Descripción
Esta celda ejecuta el motor de reglas de negocio avanzado. Realiza el análisis de topes de eventos (frecuencia de uso por póliza) comparando contra el histórico de 12 meses, aplica las reglas de "Nuevo vs Viejo Clausulado", y valida matemáticamente que los costos reportados coincidan con las tarifas pactadas (Banderazo, KM, Espera y Comisión).

Entrada
- **df_servicios_validos (DataFrame)**: Servicios que superaron el cruce de cartera en el Chunk 2.
- **tdp_core_IM_SAS.AUX_VALORSERVICIO_ASIST_AUTOS (tabla)**: Matriz maestra de tarifas, límites de eventos y comisiones.
- **tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS (tabla)**: Histórico de asistencias procesadas para control de topes.

Salida
- **df_servicios_validos_ok (DataFrame)**: Registros finales aprobados técnica y financieramente.
- **df_excede_topes (DataFrame)**: Registros rechazados por superar el límite de eventos contratados.
- **df_excede_costo (DataFrame)**: Registros rechazados por inconsistencias en el valor cobrado vs. el cálculo Allianz.
- **df_inconsistencias_total (DataFrame)**: Consolidado final de todos los errores detectados en el proceso (duplicados, incompletos, vigencia, catálogo, topes y costos).
"""

try:
    logger.info("Iniciando Chunk 3: Análisis de Topes y Validación de Costos (Versión Robusta)")

    # 1. Extraemos con un filtro de seguridad (por ejemplo, solo registros activos si existe la columna)
    logger.info("Cargando tarifas y datos históricos (12 meses atrás)...")

    query = f"""SELECT SERVICIO, SUBSERVICIO, LOCAL_CARRETERO, ESTATUS, LIVIANO_PESADO, TIPOFALLA, BANDERAZO, 
                VALOR_KM_RECORRIDO, VALOR_HORA_DE_ESPERA, VALOR_COMISION, MAXVALORSERVICIO, 
                MAXVALOR_NUEVOCLAUSULADO, LIMITE_DE_EVENTOS, N
                FROM tdp_core_IM_SAS.AUX_VALORSERVICIO_ASIST_AUTOS WHERE 1=1"""

    df_param_raw = consulta_extraccion_pool(query)

    # Extraemos y eliminamos duplicados de la tabla de tarifas
    # Aplicamos una ventana para elegir el mejor registro por cada Servicio/TipoFalla
    # Asumo que 'N' o alguna columna indica la versión más reciente
    window_param = Window.partitionBy("SERVICIO", "TIPOFALLA").orderBy(desc("N"))

    df_param_tarifas = df_param_raw.withColumn("rn", row_number().over(window_param)) \
        .filter(col("rn") == 1) \
        .drop("rn")

    logger.info(f"Registros tdp_core_IM_SAS.AUX_VALORSERVICIO_ASIST_AUTOS (Saneados): {df_param_tarifas.count()}")
    
    fecha_corte_hist = (datetime.date.today() - relativedelta(months=12)).replace(day=1).strftime("%Y-%m-%d")
    
    # Extraemos histórico y eliminamos duplicados por Expediente para no falsear los topes
    df_historico_raw = consulta_extraccion_pool(
        f"""SELECT MES, CUENTA, EXPEDIENTE, TIPO_SERVICIO, PLACA, SERVICIO, SUBSERVICIO, CODIGOSERVICIO,
          TIPOCONDUCTOR, FECHA_APERTURA, HORA_APERTURA, FECHA_REGISTRO, HORA_REGISTRO, FECHA_ASIGNACION,
          HORA_ASIGNACION, FECHA_LLEGADA_PROVEEDOR, HORA_LLEGADA_PROVEEDOR, NOMBRE_ASEGURADO, CEDULA,
          ESTATUS, CIUDAD_DE_ORIGEN, DIRECCION_DE_ORIGEN, CIUDAD_DE_DESTINO, DIRECCION_DE_DESTINO, CIUDAD_RESIDENCIA_CIRCULACION,
          CIUDAD_ORIGEN_SERVICIO, TIPOFALLA, KILOMETRAJE, HORAS_DE_ESPERA, TURNOS_EN_CUSTODIA, LOCAL_CARRETERO, BANDERAZO,
          FEE, VALOR_TOTAL, FRONT, POLIZA, APLICA, COD_SERVICIO, MODELO, SUCURSAL, COD_FASECOLDA, FEFECTO, FTERM, FANUL, FIANUL,
          MOTANUL, FECHVTO, LIVIANO_PESADO, RAMO, MARCA, CLASE, TIPO, VERSION, BANDERAZO_ALLIANZ, VALOR_KM_RECORRIDO, VALOR_HORA_DE_ESPERA,
          VALOR_COMISION, VALORTOTAL_ALLIANZ 
          FROM tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS WHERE Fecha_Apertura >= '{fecha_corte_hist}'"""
    ).dropDuplicates()

    logger.info(f"Registros tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS: {df_historico_raw.count()}")
    
    logger.info(f"Histórico cargado y saneado (sin duplicados): {df_historico_raw.count()}")
    
    df_nuevos_marcado = df_servicios_validos.withColumn("Origen", lit("Base Nueva"))
    df_hist_marcado = df_historico_raw.withColumn("Origen", lit("Historico"))
    
    # Unión y eliminación de duplicados entre histórico y lo nuevo (priorizando lo nuevo)
    df_union_hist = df_nuevos_marcado.unionByName(df_hist_marcado, allowMissingColumns=True)
    df_consol_unicos = df_union_hist.withColumn("rn_exp", row_number().over(
        Window.partitionBy("Expediente").orderBy(when(col("Origen") == "Base Nueva", 0).otherwise(1))
    )).filter(col("rn_exp") == 1).drop("rn_exp")

    # 2. Análisis de Topes
    logger.info("Clasificando reglas de topes (Old vs New Clause)...")
    df_analisis_base = df_consol_unicos.withColumn("ReglaTopes", 
        when(col("Fecha_Inicio") <= lit("2024-08-07"), lit("Old")).otherwise(lit("New")))

    cols_agrupa_topes = ["Poliza", "Aplica", "Servicio", "TipoFalla", "Vigencia", "ReglaTopes"]
    df_conteo_servicios = df_analisis_base.groupBy(*cols_agrupa_topes).agg(
        count("Vigencia").alias("Q_Servicio"),
        sum(when(col("Origen") == "Base Nueva", 1).otherwise(0)).alias("Count_New"),
        sum(when(col("Origen") == "Historico", 1).otherwise(0)).alias("Count_Old")
    )

    # 3. Cruce con Límites de Contrato
    logger.info("Cruzando con tabla de límites de eventos...")
    df_demanda_tarifada = df_conteo_servicios.alias("dem").join(
        df_param_tarifas.alias("tar"),
        (col("dem.Servicio") == col("tar.SERVICIO")) & (col("dem.TipoFalla") == col("tar.TIPOFALLA")),
        "left"
    ).select("dem.*", "tar.LIMITE_DE_EVENTOS", "tar.MAXVALORSERVICIO", "tar.MAXVALOR_NUEVOCLAUSULADO", 
             "tar.BANDERAZO", "tar.VALOR_KM_RECORRIDO", "tar.VALOR_HORA_DE_ESPERA", "tar.VALOR_COMISION")

    df_status_topes = df_demanda_tarifada.withColumn("Exclusion", coalesce(col("Q_Servicio") - col("LIMITE_DE_EVENTOS"), lit(0))) \
        .withColumn("Tarifa_Max_Permitida", when(col("ReglaTopes") == "Old", col("MAXVALORSERVICIO")).otherwise(col("MAXVALOR_NUEVOCLAUSULADO")))

    # 4. Marcación a nivel de registro
    logger.info("Aplicando marcación de exceso de servicios...")
    df_final_con_topes = df_nuevos_marcado.alias("ind").join(
        df_status_topes.alias("top"),
        on=["Poliza", "Aplica", "Servicio", "TipoFalla", "Vigencia"],
        how="left"
    ).select(col("ind.*"), col("top.ReglaTopes"), col("top.Exclusion"), col("top.Tarifa_Max_Permitida"), col("top.Q_Servicio"),
             col("top.BANDERAZO"), col("top.VALOR_KM_RECORRIDO"), col("top.VALOR_HORA_DE_ESPERA"), col("top.VALOR_COMISION"))

    window_exceso = Window.partitionBy("Poliza", "Aplica", "Servicio", "TipoFalla", "Vigencia").orderBy(col("Fecha_Apertura").desc())
    df_marcado_exceso = df_final_con_topes.withColumn("Fila_Grupo", row_number().over(window_exceso))

    # Filtrar excesos (registros rechazados)
    df_excede_topes = df_marcado_exceso.filter((col("Exclusion") > 0) & (col("Fila_Grupo") <= col("Exclusion"))) \
        .withColumn("Inconsistencia", lit("EXCEDE_CANTIDAD_SERVICIOS"))

    df_pre_ok = df_marcado_exceso.filter(~((col("Exclusion") > 0) & (col("Fila_Grupo") <= col("Exclusion"))))
    
    logger.info(f"Registros rechazados por Topes de Eventos: {df_excede_topes.count()}")

    # 5. Validación de Costos (CORREGIDO ERROR x10 Y TOPES ILIMITADOS)
    logger.info("Validando costos (Banderazo + KM + Espera)...")

    # 1. Realizamos el cálculo sobre df_pre_ok y guardamos en df_validacion_costos
    # NOTA: Dividimos Valor_Total_Num entre 10 para corregir el error de origen
    df_validacion_costos = df_pre_ok.withColumn("ValorTotal_Allianz",
        round((coalesce(col("BANDERAZO"), lit(0)) + 
            (coalesce(col("Kilometraje"), lit(0)) * coalesce(col("VALOR_KM_RECORRIDO"), lit(0))) + 
            (coalesce(col("Horas_de_espera"), lit(0)) * coalesce(col("VALOR_HORA_DE_ESPERA"), lit(0)))) * coalesce(col("VALOR_COMISION"), lit(1)))
    ).withColumn("Valor_Total_Num_Real", col("Valor_Total_Num") / 10)

    # 2. Definimos las condiciones
    # Tope ilimitado (permite Null y 0) evaluado contra el valor real (sin el cero de más)
    cond_excede_valor = (col("Tarifa_Max_Permitida").isNotNull()) & \
                        (col("Tarifa_Max_Permitida") > 0) & \
                        (col("Valor_Total_Num_Real") > col("Tarifa_Max_Permitida"))

    # Comparamos el Valor Real (aplicándole la comisión y redondeo) contra el ValorTotal_Allianz para que sea Peras vs Peras
    cond_excede_calc = (col("ValorTotal_Allianz") > 0) & \
                    (round(col("Valor_Total_Num_Real") * coalesce(col("VALOR_COMISION"), lit(1))) > col("ValorTotal_Allianz")) & \
                    (col("Tipo_Servicio") != "REEMBOLSO")

    # 3. FILTRO: Evaluamos las inconsistencias usando df_validacion_costos
    df_excede_costo = df_validacion_costos.filter(coalesce(cond_excede_valor, lit(False)) | coalesce(cond_excede_calc, lit(False))) \
        .withColumn("Inconsistencia", when(cond_excede_valor, lit("EXCEDE_VALOR_SERVICIO")).otherwise(lit("EXCEDE_VALOR_CALC_ALLIANZ")))

    # 4. Los OK
    df_servicios_validos_ok = df_validacion_costos.filter(~(coalesce(cond_excede_valor, lit(False)) | coalesce(cond_excede_calc, lit(False))))

    logger.info(f"Registros rechazados por Exceso de Costo: {df_excede_costo.count()}")

    # 6. Consolidación Final con Mitigación de NameError
    logger.info("Consolidando todas las inconsistencias detectadas...")
    
    # Lista de tuplas (Nombre, DataFrame) usando locals().get para seguridad
    potenciales_inconsistencias = [
        ("Repetidos", locals().get('df_servicios_repetidos')),
        ("Incompletos", locals().get('df_servicios_incompletos')),
        ("No Vigentes", locals().get('df_novigentes')),
        ("No Renovados", locals().get('df_norenovadas')),
        ("Inválidos Catálogo", locals().get('df_servicios_invalidos')),
        ("Excede Topes", df_excede_topes),
        ("Excede Costo", df_excede_costo)
    ]

    df_inconsistencias_total = None

    for nombre, df_error in potenciales_inconsistencias:
        if df_error is not None:
            if df_inconsistencias_total is None:
                df_inconsistencias_total = df_error
            else:
                df_inconsistencias_total = df_inconsistencias_total.unionByName(df_error, allowMissingColumns=True)
    
    # Si no hubo errores en ningún paso, creamos el DF de inconsistencias vacío pero con estructura
    if df_inconsistencias_total is None:
        df_inconsistencias_total = spark.createDataFrame([], df_asist_prep.schema).withColumn("Inconsistencia", lit(""))

    logger.info(f"--- RESUMEN FINAL ---")
    logger.info(f"Total Servicios OK: {df_servicios_validos_ok.count()}")
    logger.info(f"Total Inconsistencias: {df_inconsistencias_total.count()}")

except Exception as e:
    logger.error(f"Error crítico en Chunk 3: {str(e)}")
    raise e

In [151]:
"""
## Descripción
Esta celda finaliza el proceso realizando la limpieza e inserción de datos en las tablas históricas (Idempotencia) y la generación de los tres reportes finales en formato CSV. Además, calcula el indicador de siniestros en periodo de anulación (IND_PREST_ANUL) y consolida la información para Siniestros Instrumentales.

Entrada
- **df_servicios_validos_ok (DataFrame)**: Registros aprobados listos para pago.
- **df_inconsistencias_total (DataFrame)**: Consolidado de todos los registros rechazados durante el flujo.
- **tdp_core_IM_SAS.AUX_SUC_MEDIADOR_ASIST_AUTOS (tabla)**: Catálogo de mediadores por sucursal para siniestros instrumentales.

Salida
- **Persistencia SQL Pool**: Actualización de la tabla `tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS`.
- **Asistencias_OK_..._AAAAMM.csv**: Reporte detallado de servicios aprobados con cálculos Allianz.
- **Inconsistencias_..._AAAAMM.csv**: Reporte de errores con descripción del motivo del rechazo.
- **Siniestros_Instrumentales_..._AAAAMM.csv**: Resumen contable por ramo, sucursal y mediador.
"""

try:
    logger.info("Iniciando Chunk 4: Almacenamiento y Generación de Reportes")

    # 1. Validación de Coincidencia de Periodo y Configuración Técnica
    # -------------------------------------------------------------------------
    # Obtenemos el mes que viene en los datos para asegurar que coincide con el mes de ejecución
    row_mes_data = df_servicios_validos_ok.select("Mes").distinct().first()
    fichero_aaaamm = str(row_mes_data["Mes"]).strip() if row_mes_data else None

    if fichero_aaaamm != PROCESO_AAAAMM:
        logger.warning(f"Mes del archivo ({fichero_aaaamm}) no coincide con el periodo de proceso ({PROCESO_AAAAMM}).")
    
    # Definimos la condición para IND_PREST_ANUL (Siniestros en periodo de anulación)
    cond_anul_final = (col("fanul").isNotNull()) & (col("fanul") < col("Fecha_Apertura")) & (col("Fecha_Apertura") <= col("fianul"))

    # 2. Idempotencia y Carga del Histórico (Persistencia en SQL Pool)
    # -------------------------------------------------------------------------
    # Borramos los registros previos del mismo mes en la tabla histórica antes de insertar los nuevos
    logger.info(f"Limpiando y cargando registros del mes {PROCESO_AAAAMM} en tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS")
    
    #carga_pool(
    #    df=df_servicios_validos_ok,
    #    tabla="tdp_core_IM_SAS.ASISTENCIAS_ALZ_PARTNERS",
    #    tipo_carga="DELETE",
    #    filtro_tipo_carga=f"Mes = '{PROCESO_AAAAMM}'",
    #    caso_uso="nbk_cispc157",
    #    fecha_data=fecha_ejecucion
    #)

    # 3. Formateo y Orden: ASISTENCIAS_OK (Estructura Exacta Solicitada)
    # -------------------------------------------------------------------------
    df_final_ok_rpt = df_servicios_validos_ok.withColumn("IND_PREST_ANUL", when(cond_anul_final, 1).otherwise(0)) \
        .select(
            "Mes", "Cuenta", "Expediente", "Tipo_Servicio", "Placa", "Servicio", "SubServicio", "CodigoServicio", 
            "TipoConductor", "Fecha_Apertura", "Hora_Apertura", "Fecha_Registro", "Hora_Registro", "Fecha_Asignacion", 
            "Hora_Asignacion", "Fecha_llegada_proveedor", "Hora_llegada_proveedor", "Nombre_Asegurado", "Cedula", 
            "Estatus", "Ciudad_de_Origen", "Direccion_de_Origen", "Ciudad_de_Destino", "Direccion_de_Destino", 
            "Ciudad_Residencia_Circulacion", "Ciudad_Origen_Servicio", "TipoFalla", "Kilometraje", "Horas_de_espera", 
            col("turnos_en_custodia"), col("local_carretero"), 
            "Banderazo", # Banderazo Original
            "Fee", "Valor_Total", col("FRONT").alias("Front"), 
            col("poliza").alias("POLIZA"), col("aplica").alias("APLICA"), 
            "cod_servicio", col("modelo").alias("MODELO"), col("sucursal").alias("SUCURSAL"), 
            "cod_fasecolda", col("fefecto").alias("FEFECTO"), col("fterm").alias("FTERM"), 
            col("fanul").alias("FANUL"), col("fianul").alias("FIANUL"), "motanul", 
            col("fechvto").alias("FECHVTO"), "liviano_pesado", 
            col("ramo_Num").alias("ramo"), # Mapeo de ramo_Num a ramo
            "MARCA", "CLASE", "TIPO", "VERSION", 
            col("Q_Servicio").alias("Conteo_Servicios_TipoFalla"), "ValorTotal_Allianz", 
            col("BANDERAZO").alias("Banderazo_Allianz"), # Banderazo de Tarifas
            "VALOR_KM_RECORRIDO", "VALOR_HORA_DE_ESPERA", "VALOR_COMISION", "IND_PREST_ANUL"
        )

    # 4. Formateo y Orden: INCONSISTENCIAS (Estructura Exacta Solicitada)
    # -------------------------------------------------------------------------
    df_final_inconsistencias_rpt = df_inconsistencias_total.withColumn("IND_PREST_ANUL", when(cond_anul_final, 1).otherwise(0)) \
        .select(
            "Mes", "Cuenta", "Expediente", "Tipo_Servicio", "Placa", "Servicio", "SubServicio", "CodigoServicio", 
            "TipoConductor", "Fecha_Apertura", "Hora_Apertura", "Fecha_Registro", "Hora_Registro", "Fecha_Asignacion", 
            "Hora_Asignacion", "Fecha_llegada_proveedor", "Hora_llegada_proveedor", "Nombre_Asegurado", "Cedula", 
            "Estatus", "Ciudad_de_Origen", "Direccion_de_Origen", "Ciudad_de_Destino", "Direccion_de_Destino", 
            "Ciudad_Residencia_Circulacion", "Ciudad_Origen_Servicio", "TipoFalla", "Kilometraje", "Horas_de_espera", 
            col("turnos_en_custodia"), col("local_carretero"), "Banderazo", "Fee", "Valor_Total", 
            col("FRONT").alias("Front"), "ramo_Num", 
            col("poliza").alias("POLIZA"), col("aplica").alias("APLICA"), "cod_servicio", 
            col("modelo").alias("MODELO"), col("sucursal").alias("SUCURSAL"), "cod_fasecolda", 
            col("fefecto").alias("FEFECTO"), col("fterm").alias("FTERM"), col("fanul").alias("FANUL"), 
            col("fianul").alias("FIANUL"), "motanul", col("fechvto").alias("FECHVTO"), 
            "Inconsistencia", "Desc_Inconsistencia", "IND_PREST_ANUL"
        )

    # 5. Formateo y Orden: SINIESTROS INSTRUMENTALES
    # -------------------------------------------------------------------------
    df_mediadores = consulta_extraccion_pool("SELECT SUCURSAL, MEDIADOR FROM tdp_core_IM_SAS.AUX_SUC_MEDIADOR_ASIST_AUTOS WHERE 1=1")
    
    df_final_siniestros_rpt = df_servicios_validos_ok.alias("ok").join(
        df_mediadores.alias("med"), on="SUCURSAL", how="left"
    ).groupBy(
        col("ok.ramo_Num").alias("ramo"), 
        col("ok.SUCURSAL").alias("Sucursal"), 
        col("med.MEDIADOR").alias("Mediador")
    ).agg(
        sum("Valor_Total").alias("Valor")
    ).withColumn("Modalidad", lit(0)) \
     .withColumn("FechaSiniestro", lit(first_month_date_str)) \
     .withColumn("Circunstancia", lit("Asistencia")) \
     .withColumn("Causa", lit("Asistencia")) \
     .withColumn("Poliza", lit(1)) \
     .withColumn("Matricula", lit("999ZZZ")) \
     .withColumn("Garantia", lit("Asistencia")) \
     .withColumn("Observacion", lit("Siniestro Instrumental para los gastos del mes")) \
     .select(
        "ramo", "Sucursal", "Mediador", "Modalidad", "FechaSiniestro", "Circunstancia", 
        "Causa", "Poliza", "Matricula", "Garantia", "Valor", "Observacion"
    )

    # 6. Generación de Archivos CSV
    # -------------------------------------------------------------------------
    for df_rpt, nombre_base in [
        (df_final_ok_rpt, "Asistencias_OK_Validaciones_AllianzPartners"),
        (df_final_inconsistencias_rpt, "Inconsistencias_Asistencias_AllianzPartners"),
        (df_final_siniestros_rpt, "Siniestros_Instrumentales_Asistencia_AllianzPartners")
    ]:
        genera_informe(
            df=df_rpt,
            caso_uso="nbk_cispc157",
            nombre_archivo=f"{nombre_base}_{PROCESO_AAAAMM}.csv",
            delimitador=";",
            cabecera=True,
            hoja="N/A"
        )

    logger.info("=== PROCESO CISPC157 FINALIZADO EXITOSAMENTE ===")

except Exception as e:
    logger.error(f"Error crítico en el Chunk 4: {str(e)}")
    raise e
 